# 11: Does the benefit sit where severity sits? (EXPLORATORY)

The confirmed primary lumps grade 3 and grade 4 together. Clinically they are not equivalent:
a missed proliferative DR (grade 4) is the case that blinds people. This notebook asks whether
the ensemble benefit is concentrated in the grades that carry the severity weight.

Fresh set, 25% matched referral, rank OR-gate, 55 pairs x 9 seed combinations, patient-clustered
bootstrap. All exploratory; the pre-specified primary is unchanged.

1. **Grade-stratified benefit.** Net misses avoided per pair for grade 4, grade 3 and grade 2
   separately, with intervals, and per 1,000 patients screened.
2. **Headroom.** Each grade's marginal miss rate: models cannot rescue what they rarely miss.
3. **Grade-stratified dependence.** Joint miss rate versus the independence benchmark, by grade.
   Do models fail together more on PDR, or less?
4. **Where the weighted cost advantage comes from.** The 500/200/15 index split by grade, so it
   is visible whether the weighted benefit is carried by PDR or by moderate NPDR volume.
5. **Clinical translation**, with the conversion stated explicitly rather than buried.

CPU runtime, about 15 minutes. Run cells 1-5.

In [ ]:
# Cell 1: load fresh predictions
import io, os, json, itertools, datetime, time
import numpy as np, pandas as pd
from scipy.stats import rankdata
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
svc = gbuild('drive', 'v3')
F_PRED='1nCloJ5o9veZb2xJL_2EjdVHQXN31jy4w'; F_RES='1-HT4ONiwIfuBtC19s7QkDd8Euklh0Xon'; F_FIG='18eyHRWuqFcUrYiL8nPtpIalV7CreLkMK'
def dl(fid,p):
    with io.FileIO(p,'wb') as fh:
        d=MediaIoBaseDownload(fh, svc.files().get_media(fileId=fid)); done=False
        while not done: _,done=d.next_chunk()
def find_id(n,f):
    r=svc.files().list(q=f"'{f}' in parents and name='{n}' and trashed=false", fields='files(id)').execute().get('files',[])
    return r[0]['id'] if r else None
def up(p,f,m):
    svc.files().create(body={'name':os.path.basename(p),'parents':[f]}, media_body=MediaFileUpload(p,mimetype=m), fields='id').execute()
    print('  uploaded', os.path.basename(p))

ARCHS=['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class',
       'resnet50_binary','resnet50_5class','vit_base_binary','vit_base_5class',
       'retfound_binary','retfound_5class','retfound_adversarial']
SEEDS=[0,1,2]; PAIRS=list(itertools.combinations(ARCHS,2)); COMBOS=[(i,j) for i in SEEDS for j in SEEDS]
def tag(k): return '' if k==0 else f'_seed{k}'
BURDEN=0.25; WGT={4:500,3:200,2:15}

P={}; ref=None
for a in ARCHS:
    for k in SEEDS:
        fn=f'eyepacs_predictions_fresh_{a}{tag(k)}_v2.csv'
        if not os.path.exists(fn): dl(find_id(fn,F_PRED), fn)
        d=pd.read_csv(fn)
        if ref is None: ref=d[['image_id','patient_id','true_grade','true_label']].reset_index(drop=True)
        P[(a,k)]=d['predicted_probability'].to_numpy(np.float64)
GRD=ref['true_grade'].to_numpy(int); LAB=ref['true_label'].to_numpy(int)
PID=ref['patient_id'].astype(str).to_numpy(); N=len(GRD)
PATS=pd.unique(PID); PAT_IMGS=[np.where(PID==p)[0] for p in PATS]
print(f'fresh: {N} images, {len(PATS)} patients | grade counts:',
      {g:int((GRD==g).sum()) for g in range(5)})

In [ ]:
# Cell 2: grade-stratified benefit, headroom and dependence (point estimates)
def referred(p,R):
    m=np.zeros(len(p),bool); m[np.argsort(-p,kind='stable')[:R]]=True; return m

def analyse(idx):
    g=GRD[idx]; n=len(idx); R=int(round(BURDEN*n))
    masks={gr:(g==gr) for gr in (2,3,4)}
    sub={key:p[idx] for key,p in P.items()}
    REF={key:referred(v,R) for key,v in sub.items()}
    RK={key:rankdata(v)/n for key,v in sub.items()}
    acc={gr:[] for gr in (2,3,4)}; dep={gr:[] for gr in (2,3,4)}; marg={gr:[] for gr in (2,3,4)}
    wcost=[]
    for a,b in PAIRS:
        for i,j in COMBOS:
            orf=referred(np.maximum(RK[(a,i)],RK[(b,j)]),R)
            missA=~REF[(a,i)]; missB=~REF[(b,j)]; missOR=~orf
            # 'better constituent' fixed once, by total weighted miss cost, so grades are comparable
            cA=sum(WGT[gr]*(missA&masks[gr]).sum() for gr in (2,3,4))
            cB=sum(WGT[gr]*(missB&masks[gr]).sum() for gr in (2,3,4))
            better=missA if cA<=cB else missB
            for gr in (2,3,4):
                acc[gr].append(int((better&masks[gr]).sum()) - int((missOR&masks[gr]).sum()))
                marg[gr].append((missA&masks[gr]).sum()/max(1,masks[gr].sum()))
                joint=(missA&missB&masks[gr]).sum()/max(1,masks[gr].sum())
                indep=((missA&masks[gr]).sum()/max(1,masks[gr].sum()))*((missB&masks[gr]).sum()/max(1,masks[gr].sum()))
                dep[gr].append(joint-indep)
            wcost.append(sum(WGT[gr]*((better&masks[gr]).sum()-(missOR&masks[gr]).sum()) for gr in (2,3,4)))
    return ({gr:float(np.mean(acc[gr])) for gr in (2,3,4)},
            {gr:float(np.mean(marg[gr])) for gr in (2,3,4)},
            {gr:float(np.mean(dep[gr])) for gr in (2,3,4)},
            float(np.mean(wcost)))

t0=time.time(); AVOID,MARG,DEP,WC=analyse(np.arange(N))
print(f'computed in {time.time()-t0:.0f}s\n')
rows=[]
for gr,name in ((4,'grade 4 (PDR)'),(3,'grade 3 (severe NPDR)'),(2,'grade 2 (moderate NPDR)')):
    rows.append({'grade':name,'eyes_in_set':int((GRD==gr).sum()),
                 'mean_marginal_miss_rate':round(MARG[gr],4),
                 'net_misses_avoided_per_pair':round(AVOID[gr],2),
                 'per_1000_patients':round(AVOID[gr]/len(PATS)*1000,3),
                 'weighted_contribution':round(AVOID[gr]*WGT[gr],1),
                 'share_of_weighted_benefit':None,
                 'joint_miss_excess_over_independence':round(DEP[gr],4)})
tot=sum(r['weighted_contribution'] for r in rows)
for r in rows: r['share_of_weighted_benefit']=round(r['weighted_contribution']/tot,3) if tot else None
T=pd.DataFrame(rows); print(T.to_string(index=False))
print(f'\nmean weighted cost advantage per pair (500/200/15): {WC:.1f}')

In [ ]:
# Cell 3: patient-clustered bootstrap for the grade-stratified benefit
rng=np.random.default_rng(20260922); B=400; boots=[]
t0=time.time()
for b in range(B):
    idx=np.concatenate([PAT_IMGS[i] for i in rng.integers(0,len(PATS),len(PATS))])
    av,_,_,wc=analyse(idx)
    boots.append([av[4],av[3],av[2],wc])
    if (b+1)%50==0: print(f'  {b+1}/{B} ({time.time()-t0:.0f}s)')
boots=np.array(boots)
CI={'grade4':[round(float(x),2) for x in np.percentile(boots[:,0],[2.5,97.5])],
    'grade3':[round(float(x),2) for x in np.percentile(boots[:,1],[2.5,97.5])],
    'grade2':[round(float(x),2) for x in np.percentile(boots[:,2],[2.5,97.5])],
    'weighted_cost':[round(float(x),1) for x in np.percentile(boots[:,3],[2.5,97.5])]}
print('\nnet misses avoided per pair, 95% patient-clustered intervals:')
print(f"  grade 4 (PDR):           {AVOID[4]:.2f}  {CI['grade4']}")
print(f"  grade 3 (severe NPDR):   {AVOID[3]:.2f}  {CI['grade3']}")
print(f"  grade 2 (moderate NPDR): {AVOID[2]:.2f}  {CI['grade2']}")
print(f"  weighted cost advantage: {WC:.1f}  {CI['weighted_cost']}")

In [ ]:
# Cell 4: clinical translation, conversion stated explicitly
per1000={gr:AVOID[gr]/len(PATS)*1000 for gr in (2,3,4)}
pdr_per_100k=per1000[4]*100
CLIN={
 'screened_patients_in_fresh_set':int(len(PATS)),
 'pdr_misses_avoided_per_1000_screened':round(per1000[4],3),
 'pdr_misses_avoided_per_100000_screened':round(pdr_per_100k,1),
 'severe_npdr_misses_avoided_per_1000':round(per1000[3],3),
 'moderate_npdr_misses_avoided_per_1000':round(per1000[2],3),
 'number_needed_to_screen_per_pdr_miss_avoided': round(1000/per1000[4]) if per1000[4]>0 else None,
 'conversion_note': ('A missed PDR at one screening round is not equivalent to blindness: patients are '
                     'rescreened, present symptomatically, or are treated later. Any QALY figure derived '
                     'from these counts is an upper bound on benefit, not an estimate.'),
 'upper_bound_qaly_per_100k_if_every_missed_pdr_led_to_blindness': round(pdr_per_100k*6.42,1),
}
print(json.dumps(CLIN, indent=2))
print('\nReading: if the grade-4 interval excludes zero, severity weighting is doing real work and')
print('the benefit sits where the clinical cost sits. If it straddles zero while grade 3 does not,')
print('the benefit is in severe NPDR, and the PDR claim cannot be made from these data.')

In [ ]:
# Cell 5: figure and save
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(12,4.4))
g=[4,3,2]; names=['Grade 4\n(PDR)','Grade 3\n(severe NPDR)','Grade 2\n(moderate NPDR)']
vals=[AVOID[x] for x in g]; los=[CI[f'grade{x}'][0] for x in g]; his=[CI[f'grade{x}'][1] for x in g]
ax[0].bar(names, vals, color=['C3','C1','C0'])
ax[0].errorbar(names, vals, yerr=[np.array(vals)-np.array(los), np.array(his)-np.array(vals)],
               fmt='none', ecolor='k', capsize=4)
ax[0].axhline(0,color='grey',lw=0.8); ax[0].set_ylabel('Net misses avoided per pair')
ax[0].set_title('Benefit by severity grade (matched workload)')
shares=[AVOID[x]*WGT[x] for x in g]
ax[1].bar(names, shares, color=['C3','C1','C0']); ax[1].axhline(0,color='grey',lw=0.8)
ax[1].set_ylabel('Contribution to weighted cost advantage'); ax[1].set_title('Where the severity-weighted benefit comes from')
fig.suptitle('Is the ensemble benefit where the severity is? (exploratory, fresh set)')
fig.tight_layout(); fig.savefig('benefit_by_severity_v3.png', dpi=200)

OUT={'generated_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
     'status':'EXPLORATORY; the pre-specified primary is unchanged',
     'burden':BURDEN,'weights':WGT,'by_grade':T.to_dict(orient='records'),
     'intervals':CI,'weighted_cost_advantage_per_pair':round(WC,1),'clinical':CLIN}
with open('benefit_by_severity_v3.json','w') as f: json.dump(OUT,f,indent=2,default=str)
T.to_csv('benefit_by_severity_v3.csv', index=False)
up('benefit_by_severity_v3.json',F_RES,'application/json'); up('benefit_by_severity_v3.csv',F_RES,'text/csv')
up('benefit_by_severity_v3.png',F_FIG,'image/png')
print('saved')